# ** Avoid **
# This note book deos not make sense

# 2_shared_multiCore_XLM-R_diffWeightsBoth

## Confusion Matrix sub and main together

In [1]:
import json
from pathlib import Path

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
OUTPUT_DIR = Path(
    r"C:\Users\alrazz\Downloads\shared_multiCore_XLM-R_diffWeightsBoth"
)



In [3]:
LOGITS_FILE = (
    OUTPUT_DIR / "test_logits.npy"
)

LABELS_FILE = (
    OUTPUT_DIR / "test_labels.npy"
)

LABEL_NAMES_FILE = (
    OUTPUT_DIR / "hierarchical_labels.json"
)


In [4]:
CM_DIR = (
    OUTPUT_DIR / "confusion_matrices"
)

CM_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

In [5]:
print("\n" + "=" * 100)
print("LOADING TEST RESULTS")
print("=" * 100)


test_logits = np.load(
    LOGITS_FILE
)

test_labels = np.load(
    LABELS_FILE
)


with open(
    LABEL_NAMES_FILE,
    "r",
    encoding="utf-8",
) as f:

    hierarchical_labels = json.load(f)


print(
    "Logits shape:",
    test_logits.shape,
)

print(
    "Labels shape:",
    test_labels.shape,
)

print(
    "Number of labels:",
    len(hierarchical_labels),
)



LOADING TEST RESULTS
Logits shape: (633, 25)
Labels shape: (633, 25)
Number of labels: 25


In [6]:
if test_logits.shape != test_labels.shape:

    raise ValueError(
        "test_logits and test_labels have different shapes:\n"
        f"logits = {test_logits.shape}\n"
        f"labels = {test_labels.shape}"
    )


if test_logits.shape[1] != len(
    hierarchical_labels
):

    raise ValueError(
        "Number of labels does not match "
        "logits columns."
    )


NUM_LABELS = len(
    hierarchical_labels
)


In [7]:
NUM_PARENTS = 9

parents = hierarchical_labels[
    :NUM_PARENTS
]

children = hierarchical_labels[
    NUM_PARENTS:
]


print("\nParents:")
print(parents)

print("\nChildren:")
print(children)



Parents:
['MT', 'LY', 'SP', 'ID', 'NA', 'HI', 'IN', 'OP', 'IP']

Children:
['it', 'ne', 'sr', 'nb', 're', 'en', 'ra', 'dtp', 'fi', 'lt', 'rv', 'ob', 'rs', 'av', 'ds', 'ed']


In [8]:
# SIGMOID

def sigmoid(x):

    return 1.0 / (
        1.0 + np.exp(-x)
    )


test_probabilities = sigmoid(
    test_logits
)


In [9]:

def make_predictions(
    probabilities,
    thresholds,
):

    """
    probabilities:
        shape = [N, 25]

    thresholds:
        either

        - single float, e.g. 0.5

        OR

        - array/list of 25 thresholds
    """

    thresholds = np.asarray(
        thresholds,
        dtype=np.float32,
    )

    if thresholds.ndim == 0:

        thresholds = np.repeat(
            thresholds,
            NUM_LABELS,
        )

    if len(thresholds) != NUM_LABELS:

        raise ValueError(
            "Number of thresholds must equal "
            f"{NUM_LABELS}."
        )

    predictions = (
        probabilities
        >= thresholds.reshape(1, -1)
    ).astype(np.int32)

    return predictions

In [10]:
def build_label_confusion_matrix(
    true_labels,
    predicted_labels,
):
   
    n_labels = true_labels.shape[1]

    confusion = np.zeros(
        (n_labels, n_labels),
        dtype=np.int64,
    )


    for i in range(
        true_labels.shape[0]
    ):

        true_indices = np.where(
            true_labels[i] == 1
        )[0]

        predicted_indices = np.where(
            predicted_labels[i] == 1
        )[0]


        for true_idx in true_indices:

            for pred_idx in predicted_indices:

                confusion[
                    true_idx,
                    pred_idx
                ] += 1


    return confusion


In [11]:
def confusion_to_percentages(
    confusion
):


    row_totals = (
        confusion.sum(axis=1)
    )


    percentages = np.zeros_like(
        confusion,
        dtype=np.float64,
    )


    for i in range(
        confusion.shape[0]
    ):

        if row_totals[i] > 0:

            percentages[i] = (
                confusion[i]
                / row_totals[i]
                * 100.0
            )


    return percentages

In [12]:
def save_matrices(
    confusion,
    percentages,
    prefix,
):

    # --------------------------------------------------------
    # Counts
    # --------------------------------------------------------

    count_df = pd.DataFrame(
        confusion,
        index=hierarchical_labels,
        columns=hierarchical_labels,
    )


    count_file = (
        CM_DIR
        / f"{prefix}_counts.csv"
    )


    count_df.to_csv(
        count_file,
        encoding="utf-8-sig",
    )


    # --------------------------------------------------------
    # Percentages
    # --------------------------------------------------------

    percent_df = pd.DataFrame(
        percentages,
        index=hierarchical_labels,
        columns=hierarchical_labels,
    )


    percent_file = (
        CM_DIR
        / f"{prefix}_percentages.csv"
    )


    percent_df.to_csv(
        percent_file,
        encoding="utf-8-sig",
        float_format="%.4f",
    )


    return count_df, percent_df

In [13]:
def create_annotations(
    confusion,
    percentages,
):

    annotations = np.empty(
        confusion.shape,
        dtype=object,
    )


    for i in range(
        confusion.shape[0]
    ):

        for j in range(
            confusion.shape[1]
        ):

            annotations[i, j] = (
                f"{confusion[i, j]:,}\n"
                f"{percentages[i, j]:.1f}%"
            )


    return annotations

In [14]:
def plot_confusion_matrix(
    confusion,
    percentages,
    prefix,
    title,
):

    annotations = create_annotations(
        confusion,
        percentages,
    )


    # --------------------------------------------------------
    # Figure size
    #
    # 25 labels require a large figure.
    # --------------------------------------------------------

    fig, ax = plt.subplots(
        figsize=(24, 21)
    )


    # --------------------------------------------------------
    # Heatmap
    # --------------------------------------------------------

    sns.heatmap(
        percentages,

        annot=annotations,

        fmt="",

        cmap="Blues",

        linewidths=0.5,

        linecolor="white",

        xticklabels=hierarchical_labels,

        yticklabels=hierarchical_labels,

        cbar_kws={
            "label":
            "Percentage (%)"
        },

        ax=ax,
    )


    # --------------------------------------------------------
    # Labels
    # --------------------------------------------------------

    ax.set_xlabel(
        "Predicted Label",
        fontsize=15,
        fontweight="bold",
    )


    ax.set_ylabel(
        "True Label",
        fontsize=15,
        fontweight="bold",
    )


    ax.set_title(
        title,
        fontsize=18,
        fontweight="bold",
        pad=20,
    )


    # --------------------------------------------------------
    # Tick labels
    # --------------------------------------------------------

    ax.set_xticklabels(
        hierarchical_labels,
        rotation=45,
        ha="right",
        fontsize=10,
    )


    ax.set_yticklabels(
        hierarchical_labels,
        rotation=0,
        fontsize=10,
    )


    # --------------------------------------------------------
    # Layout
    # --------------------------------------------------------

    plt.tight_layout()


    # --------------------------------------------------------
    # Save PNG
    # --------------------------------------------------------

    png_file = (
        CM_DIR
        / f"{prefix}.png"
    )


    plt.savefig(
        png_file,
        dpi=400,
        bbox_inches="tight",
    )


    # --------------------------------------------------------
    # Save PDF
    #
    # Useful for papers because it is vector-based.
    # --------------------------------------------------------

    pdf_file = (
        CM_DIR
        / f"{prefix}.pdf"
    )


    plt.savefig(
        pdf_file,
        bbox_inches="tight",
    )


    plt.close()


    print(
        "\nSaved:"
    )

    print(
        png_file
    )

    print(
        pdf_file
    )


In [15]:
def create_confusion_matrix(
    predictions,
    prefix,
    title,
):

    print("\n" + "-" * 100)

    print(
        "Creating:",
        title
    )


    # --------------------------------------------------------
    # Build count matrix
    # --------------------------------------------------------

    confusion = (
        build_label_confusion_matrix(
            test_labels,
            predictions,
        )
    )


    # --------------------------------------------------------
    # Convert to percentages
    # --------------------------------------------------------

    percentages = (
        confusion_to_percentages(
            confusion
        )
    )


    # --------------------------------------------------------
    # Save CSV
    # --------------------------------------------------------

    save_matrices(
        confusion,
        percentages,
        prefix,
    )


    # --------------------------------------------------------
    # Plot
    # --------------------------------------------------------

    plot_confusion_matrix(
        confusion,
        percentages,
        prefix,
        title,
    )


    return (
        confusion,
        percentages,
    )

In [16]:
# 1. ORIGINAL 0.5 THRESHOLD


print("\n" + "=" * 100)
print("1. ORIGINAL 0.5 THRESHOLD")
print("=" * 100)


predictions_05 = make_predictions(
    test_probabilities,
    0.5,
)


confusion_05, percentages_05 = (
    create_confusion_matrix(

        predictions=predictions_05,

        prefix="confusion_0.5",

        title=(
            "Hierarchical Multilabel "
            "Label Confusion Matrix "
            "(Threshold = 0.50)"
        ),
    )
)


1. ORIGINAL 0.5 THRESHOLD

----------------------------------------------------------------------------------------------------
Creating: Hierarchical Multilabel Label Confusion Matrix (Threshold = 0.50)

Saved:
C:\Users\alrazz\Downloads\shared_multiCore_XLM-R_diffWeightsBoth\confusion_matrices\confusion_0.5.png
C:\Users\alrazz\Downloads\shared_multiCore_XLM-R_diffWeightsBoth\confusion_matrices\confusion_0.5.pdf


In [17]:
# GLOBAL THRESHOLD TUNING

GLOBAL_TUNED_THRESHOLD = 0.35


print("\n" + "=" * 100)
print("2. GLOBAL TUNED THRESHOLD")
print("=" * 100)

print(
    "Global threshold:",
    GLOBAL_TUNED_THRESHOLD,
)


predictions_global = make_predictions(
    test_probabilities,
    GLOBAL_TUNED_THRESHOLD,
)


confusion_global, percentages_global = (
    create_confusion_matrix(

        predictions=predictions_global,

        prefix="confusion_global_tuned",

        title=(
            "Hierarchical Multilabel "
            "Label Confusion Matrix "
            f"(Global Threshold = "
            f"{GLOBAL_TUNED_THRESHOLD:.3f})"
        ),
    )
)


2. GLOBAL TUNED THRESHOLD
Global threshold: 0.35

----------------------------------------------------------------------------------------------------
Creating: Hierarchical Multilabel Label Confusion Matrix (Global Threshold = 0.350)

Saved:
C:\Users\alrazz\Downloads\shared_multiCore_XLM-R_diffWeightsBoth\confusion_matrices\confusion_global_tuned.png
C:\Users\alrazz\Downloads\shared_multiCore_XLM-R_diffWeightsBoth\confusion_matrices\confusion_global_tuned.pdf


In [18]:
#  PER-LABEL THRESHOLD TUNING

PER_LABEL_THRESHOLDS = {

"MT"  : 0.10,
"LY"  : 0.50,
"SP"  : 0.53,
"ID"  : 0.66,
"NA"  : 0.44,
"HI"  : 0.24,
"IN"  : 0.46,
"OP"  : 0.35,
"IP"  : 0.28,
"it"  : 0.43,
"ne"  : 0.28,
"sr"  : 0.39,
"nb"  : 0.26,
"re"  : 0.49,
"en"  : 0.44,
"ra"  : 0.16,
"dtp" : 0.25,
"fi"  : 0.33,
"lt"  : 0.38,
"rv"  : 0.21,
"ob"  : 0.26,
"rs"  : 0.47,
"av"  : 0.45,
"ds"  : 0.53,
"ed"  : 0.32

}


In [19]:
per_label_threshold_array = np.array(
    [
        PER_LABEL_THRESHOLDS[label]
        for label in hierarchical_labels
    ],
    dtype=np.float32,
)


print("\nPer-label thresholds:")

for label, threshold in zip(
    hierarchical_labels,
    per_label_threshold_array,
):

    print(
        f"{label:>5s}: "
        f"{threshold:.4f}"
    )


Per-label thresholds:
   MT: 0.1000
   LY: 0.5000
   SP: 0.5300
   ID: 0.6600
   NA: 0.4400
   HI: 0.2400
   IN: 0.4600
   OP: 0.3500
   IP: 0.2800
   it: 0.4300
   ne: 0.2800
   sr: 0.3900
   nb: 0.2600
   re: 0.4900
   en: 0.4400
   ra: 0.1600
  dtp: 0.2500
   fi: 0.3300
   lt: 0.3800
   rv: 0.2100
   ob: 0.2600
   rs: 0.4700
   av: 0.4500
   ds: 0.5300
   ed: 0.3200


In [20]:
predictions_per_label = (
    make_predictions(
        test_probabilities,
        per_label_threshold_array,
    )
)


confusion_per_label, percentages_per_label = (
    create_confusion_matrix(

        predictions=predictions_per_label,

        prefix="confusion_per_label_tuned",

        title=(
            "Hierarchical Multilabel "
            "Label Confusion Matrix "
            "(Per-Label Tuned Thresholds)"
        ),
    )
)




----------------------------------------------------------------------------------------------------
Creating: Hierarchical Multilabel Label Confusion Matrix (Per-Label Tuned Thresholds)

Saved:
C:\Users\alrazz\Downloads\shared_multiCore_XLM-R_diffWeightsBoth\confusion_matrices\confusion_per_label_tuned.png
C:\Users\alrazz\Downloads\shared_multiCore_XLM-R_diffWeightsBoth\confusion_matrices\confusion_per_label_tuned.pdf


## Confusion Matrix Parent Child seprately

In [25]:

def create_subset_confusion_matrix(
    true_labels,
    predicted_labels,
    label_indices,
    label_names,
    prefix,
    title,
):


    # --------------------------------------------------------
    # Extract only the desired labels
    # --------------------------------------------------------

    true_subset = (
        true_labels[:, label_indices]
    )

    predicted_subset = (
        predicted_labels[:, label_indices]
    )


    n_labels = len(
        label_names
    )


    # --------------------------------------------------------
    # Create confusion matrix
    # --------------------------------------------------------

    confusion = np.zeros(
        (n_labels, n_labels),
        dtype=np.int64,
    )


    # --------------------------------------------------------
    # Count true -> predicted label associations
    # --------------------------------------------------------

    for i in range(
        true_subset.shape[0]
    ):

        true_indices = np.where(
            true_subset[i] == 1
        )[0]

        predicted_indices = np.where(
            predicted_subset[i] == 1
        )[0]


        for true_idx in true_indices:

            for pred_idx in predicted_indices:

                confusion[
                    true_idx,
                    pred_idx
                ] += 1


    # --------------------------------------------------------
    # Row-wise percentages
    # --------------------------------------------------------

    row_totals = (
        confusion.sum(axis=1)
    )


    percentages = np.zeros_like(
        confusion,
        dtype=np.float64,
    )


    for i in range(n_labels):

        if row_totals[i] > 0:

            percentages[i] = (
                confusion[i]
                / row_totals[i]
                * 100.0
            )


    # ========================================================
    # SAVE COUNTS
    # ========================================================

    count_df = pd.DataFrame(
        confusion,
        index=label_names,
        columns=label_names,
    )


    count_file = (
        CM_DIR
        / f"{prefix}_counts.csv"
    )


    count_df.to_csv(
        count_file,
        encoding="utf-8-sig",
    )


    # ========================================================
    # SAVE PERCENTAGES
    # ========================================================

    percent_df = pd.DataFrame(
        percentages,
        index=label_names,
        columns=label_names,
    )


    percent_file = (
        CM_DIR
        / f"{prefix}_percentages.csv"
    )


    percent_df.to_csv(
        percent_file,
        encoding="utf-8-sig",
        float_format="%.4f",
    )


    # ========================================================
    # CREATE ANNOTATIONS
    # ========================================================

    annotations = np.empty(
        confusion.shape,
        dtype=object,
    )


    for i in range(n_labels):

        for j in range(n_labels):

            annotations[i, j] = (
                f"{confusion[i, j]:,}\n"
                f"{percentages[i, j]:.1f}%"
            )


    # ========================================================
    # FIGURE SIZE
    # ========================================================

    if n_labels <= 10:

        figsize = (12, 10)

    elif n_labels <= 16:

        figsize = (16, 14)

    else:

        figsize = (24, 21)


    fig, ax = plt.subplots(
        figsize=figsize
    )


    # ========================================================
    # HEATMAP
    # ========================================================

    sns.heatmap(

        percentages,

        annot=annotations,

        fmt="",

        cmap="Blues",

        linewidths=0.6,

        linecolor="white",

        xticklabels=label_names,

        yticklabels=label_names,

        cbar_kws={
            "label":
            "Percentage (%)"
        },

        ax=ax,
    )


    # ========================================================
    # AXIS LABELS
    # ========================================================

    ax.set_xlabel(
        "Predicted Label",
        fontsize=14,
        fontweight="bold",
    )


    ax.set_ylabel(
        "True Label",
        fontsize=14,
        fontweight="bold",
    )


    ax.set_title(
        title,
        fontsize=17,
        fontweight="bold",
        pad=18,
    )


    # ========================================================
    # TICK LABELS
    # ========================================================

    ax.set_xticklabels(
        label_names,
        rotation=45,
        ha="right",
        fontsize=11,
    )


    ax.set_yticklabels(
        label_names,
        rotation=0,
        fontsize=11,
    )


    # ========================================================
    # SAVE PNG
    # ========================================================

    png_file = (
        CM_DIR
        / f"{prefix}.png"
    )


    plt.tight_layout()


    plt.savefig(
        png_file,
        dpi=400,
        bbox_inches="tight",
    )


    # ========================================================
    # SAVE PDF
    # ========================================================

    pdf_file = (
        CM_DIR
        / f"{prefix}.pdf"
    )


    plt.savefig(
        pdf_file,
        bbox_inches="tight",
    )


    plt.close()


    print(
        "\nSaved:"
    )

    print(
        png_file
    )

    print(
        pdf_file
    )

    print(
        count_file
    )

    print(
        percent_file
    )


    return (
        confusion,
        percentages,
    )





In [26]:
# ============================================================
# LABEL INDICES
# ============================================================

parent_indices = list(
    range(
        len(parents)
    )
)


child_indices = list(
    range(
        len(parents),
        len(hierarchical_labels)
    )
)


In [27]:

print("\nParent indices:")
print(parent_indices)


print("\nChild indices:")
print(child_indices)


Parent indices:
[0, 1, 2, 3, 4, 5, 6, 7, 8]

Child indices:
[9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24]


In [28]:
def create_parent_child_matrices(
    predictions,
    prefix,
    threshold_description,
):

    # ========================================================
    # PARENT MATRIX
    # ========================================================

    print("\n" + "-" * 100)

    print(
        "Creating parent confusion matrix:",
        threshold_description,
    )


    parent_confusion, parent_percentages = (
        create_subset_confusion_matrix(

            true_labels=test_labels,

            predicted_labels=predictions,

            label_indices=parent_indices,

            label_names=parents,

            prefix=(
                f"{prefix}_parents"
            ),

            title=(
                "Parent Label Confusion Matrix\n"
                f"{threshold_description}"
            ),
        )
    )


    # ========================================================
    # CHILD MATRIX
    # ========================================================

    print("\n" + "-" * 100)

    print(
        "Creating child confusion matrix:",
        threshold_description,
    )


    child_confusion, child_percentages = (
        create_subset_confusion_matrix(

            true_labels=test_labels,

            predicted_labels=predictions,

            label_indices=child_indices,

            label_names=children,

            prefix=(
                f"{prefix}_children"
            ),

            title=(
                "Child Label Confusion Matrix\n"
                f"{threshold_description}"
            ),
        )
    )


    return {
        "parent_confusion":
            parent_confusion,

        "parent_percentages":
            parent_percentages,

        "child_confusion":
            child_confusion,

        "child_percentages":
            child_percentages,
    }

In [29]:
# ORIGINAL 0.5 THRESHOLD
print("\n" + "=" * 100)
print("PARENT + CHILD MATRICES — THRESHOLD 0.50")
print("=" * 100)


parent_child_05 = (
    create_parent_child_matrices(

        predictions=predictions_05,

        prefix="confusion_0.5",

        threshold_description=(
            "Threshold = 0.50"
        ),
    )
)



PARENT + CHILD MATRICES — THRESHOLD 0.50

----------------------------------------------------------------------------------------------------
Creating parent confusion matrix: Threshold = 0.50

Saved:
C:\Users\alrazz\Downloads\shared_multiCore_XLM-R_diffWeightsBoth\confusion_matrices\confusion_0.5_parents.png
C:\Users\alrazz\Downloads\shared_multiCore_XLM-R_diffWeightsBoth\confusion_matrices\confusion_0.5_parents.pdf
C:\Users\alrazz\Downloads\shared_multiCore_XLM-R_diffWeightsBoth\confusion_matrices\confusion_0.5_parents_counts.csv
C:\Users\alrazz\Downloads\shared_multiCore_XLM-R_diffWeightsBoth\confusion_matrices\confusion_0.5_parents_percentages.csv

----------------------------------------------------------------------------------------------------
Creating child confusion matrix: Threshold = 0.50

Saved:
C:\Users\alrazz\Downloads\shared_multiCore_XLM-R_diffWeightsBoth\confusion_matrices\confusion_0.5_children.png
C:\Users\alrazz\Downloads\shared_multiCore_XLM-R_diffWeightsBoth\co

In [30]:
#Global threshold
print("\n" + "=" * 100)
print("PARENT + CHILD MATRICES — GLOBAL TUNED")
print("=" * 100)


parent_child_global = (
    create_parent_child_matrices(

        predictions=predictions_global,

        prefix="confusion_global_tuned",

        threshold_description=(
            "Global Tuned Threshold = "
            f"{GLOBAL_TUNED_THRESHOLD:.3f}"
        ),
    )
)



PARENT + CHILD MATRICES — GLOBAL TUNED

----------------------------------------------------------------------------------------------------
Creating parent confusion matrix: Global Tuned Threshold = 0.350

Saved:
C:\Users\alrazz\Downloads\shared_multiCore_XLM-R_diffWeightsBoth\confusion_matrices\confusion_global_tuned_parents.png
C:\Users\alrazz\Downloads\shared_multiCore_XLM-R_diffWeightsBoth\confusion_matrices\confusion_global_tuned_parents.pdf
C:\Users\alrazz\Downloads\shared_multiCore_XLM-R_diffWeightsBoth\confusion_matrices\confusion_global_tuned_parents_counts.csv
C:\Users\alrazz\Downloads\shared_multiCore_XLM-R_diffWeightsBoth\confusion_matrices\confusion_global_tuned_parents_percentages.csv

----------------------------------------------------------------------------------------------------
Creating child confusion matrix: Global Tuned Threshold = 0.350

Saved:
C:\Users\alrazz\Downloads\shared_multiCore_XLM-R_diffWeightsBoth\confusion_matrices\confusion_global_tuned_children.

In [31]:
# PER-LABEL TUNED THRESHOLDS


print("\n" + "=" * 100)
print("PARENT + CHILD MATRICES — PER-LABEL TUNED")
print("=" * 100)


parent_child_per_label = (
    create_parent_child_matrices(

        predictions=predictions_per_label,

        prefix="confusion_per_label_tuned",

        threshold_description=(
            "Per-Label Tuned Thresholds"
        ),
    )
)



PARENT + CHILD MATRICES — PER-LABEL TUNED

----------------------------------------------------------------------------------------------------
Creating parent confusion matrix: Per-Label Tuned Thresholds

Saved:
C:\Users\alrazz\Downloads\shared_multiCore_XLM-R_diffWeightsBoth\confusion_matrices\confusion_per_label_tuned_parents.png
C:\Users\alrazz\Downloads\shared_multiCore_XLM-R_diffWeightsBoth\confusion_matrices\confusion_per_label_tuned_parents.pdf
C:\Users\alrazz\Downloads\shared_multiCore_XLM-R_diffWeightsBoth\confusion_matrices\confusion_per_label_tuned_parents_counts.csv
C:\Users\alrazz\Downloads\shared_multiCore_XLM-R_diffWeightsBoth\confusion_matrices\confusion_per_label_tuned_parents_percentages.csv

----------------------------------------------------------------------------------------------------
Creating child confusion matrix: Per-Label Tuned Thresholds

Saved:
C:\Users\alrazz\Downloads\shared_multiCore_XLM-R_diffWeightsBoth\confusion_matrices\confusion_per_label_tuned

# 4_Parent_Conditioned_Hierarchical_XLM-R_diffWeightsBoth

## Confusion Matrix sub and main together

In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

In [32]:
OUTPUT_DIR = Path(
    r"C:\Users\alrazz\Downloads\Parent_Conditioned_Hierarchica"
)



In [33]:
LOGITS_FILE = (
    OUTPUT_DIR / "test_logits.npy"
)

LABELS_FILE = (
    OUTPUT_DIR / "test_labels.npy"
)

LABEL_NAMES_FILE = (
    OUTPUT_DIR / "hierarchical_labels.json"
)


In [34]:
CM_DIR = (
    OUTPUT_DIR / "confusion_matrices"
)

CM_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

In [35]:
print("\n" + "=" * 100)
print("LOADING TEST RESULTS")
print("=" * 100)


test_logits = np.load(
    LOGITS_FILE
)

test_labels = np.load(
    LABELS_FILE
)


with open(
    LABEL_NAMES_FILE,
    "r",
    encoding="utf-8",
) as f:

    hierarchical_labels = json.load(f)


print(
    "Logits shape:",
    test_logits.shape,
)

print(
    "Labels shape:",
    test_labels.shape,
)

print(
    "Number of labels:",
    len(hierarchical_labels),
)



LOADING TEST RESULTS
Logits shape: (633, 25)
Labels shape: (633, 25)
Number of labels: 25


In [36]:
if test_logits.shape != test_labels.shape:

    raise ValueError(
        "test_logits and test_labels have different shapes:\n"
        f"logits = {test_logits.shape}\n"
        f"labels = {test_labels.shape}"
    )


if test_logits.shape[1] != len(
    hierarchical_labels
):

    raise ValueError(
        "Number of labels does not match "
        "logits columns."
    )


NUM_LABELS = len(
    hierarchical_labels
)


In [37]:
NUM_PARENTS = 9

parents = hierarchical_labels[
    :NUM_PARENTS
]

children = hierarchical_labels[
    NUM_PARENTS:
]


print("\nParents:")
print(parents)

print("\nChildren:")
print(children)



Parents:
['MT', 'LY', 'SP', 'ID', 'NA', 'HI', 'IN', 'OP', 'IP']

Children:
['it', 'ne', 'sr', 'nb', 're', 'en', 'ra', 'dtp', 'fi', 'lt', 'rv', 'ob', 'rs', 'av', 'ds', 'ed']


In [38]:
# SIGMOID

def sigmoid(x):

    return 1.0 / (
        1.0 + np.exp(-x)
    )


test_probabilities = sigmoid(
    test_logits
)


In [39]:

def make_predictions(
    probabilities,
    thresholds,
):

    """
    probabilities:
        shape = [N, 25]

    thresholds:
        either

        - single float, e.g. 0.5

        OR

        - array/list of 25 thresholds
    """

    thresholds = np.asarray(
        thresholds,
        dtype=np.float32,
    )

    if thresholds.ndim == 0:

        thresholds = np.repeat(
            thresholds,
            NUM_LABELS,
        )

    if len(thresholds) != NUM_LABELS:

        raise ValueError(
            "Number of thresholds must equal "
            f"{NUM_LABELS}."
        )

    predictions = (
        probabilities
        >= thresholds.reshape(1, -1)
    ).astype(np.int32)

    return predictions

In [40]:
def build_label_confusion_matrix(
    true_labels,
    predicted_labels,
):
   
    n_labels = true_labels.shape[1]

    confusion = np.zeros(
        (n_labels, n_labels),
        dtype=np.int64,
    )


    for i in range(
        true_labels.shape[0]
    ):

        true_indices = np.where(
            true_labels[i] == 1
        )[0]

        predicted_indices = np.where(
            predicted_labels[i] == 1
        )[0]


        for true_idx in true_indices:

            for pred_idx in predicted_indices:

                confusion[
                    true_idx,
                    pred_idx
                ] += 1


    return confusion


In [41]:
def confusion_to_percentages(
    confusion
):


    row_totals = (
        confusion.sum(axis=1)
    )


    percentages = np.zeros_like(
        confusion,
        dtype=np.float64,
    )


    for i in range(
        confusion.shape[0]
    ):

        if row_totals[i] > 0:

            percentages[i] = (
                confusion[i]
                / row_totals[i]
                * 100.0
            )


    return percentages

In [42]:
def save_matrices(
    confusion,
    percentages,
    prefix,
):

    # --------------------------------------------------------
    # Counts
    # --------------------------------------------------------

    count_df = pd.DataFrame(
        confusion,
        index=hierarchical_labels,
        columns=hierarchical_labels,
    )


    count_file = (
        CM_DIR
        / f"{prefix}_counts.csv"
    )


    count_df.to_csv(
        count_file,
        encoding="utf-8-sig",
    )


    # --------------------------------------------------------
    # Percentages
    # --------------------------------------------------------

    percent_df = pd.DataFrame(
        percentages,
        index=hierarchical_labels,
        columns=hierarchical_labels,
    )


    percent_file = (
        CM_DIR
        / f"{prefix}_percentages.csv"
    )


    percent_df.to_csv(
        percent_file,
        encoding="utf-8-sig",
        float_format="%.4f",
    )


    return count_df, percent_df

In [43]:
def create_annotations(
    confusion,
    percentages,
):

    annotations = np.empty(
        confusion.shape,
        dtype=object,
    )


    for i in range(
        confusion.shape[0]
    ):

        for j in range(
            confusion.shape[1]
        ):

            annotations[i, j] = (
                f"{confusion[i, j]:,}\n"
                f"{percentages[i, j]:.1f}%"
            )


    return annotations

In [44]:
def plot_confusion_matrix(
    confusion,
    percentages,
    prefix,
    title,
):

    annotations = create_annotations(
        confusion,
        percentages,
    )


    # --------------------------------------------------------
    # Figure size
    #
    # 25 labels require a large figure.
    # --------------------------------------------------------

    fig, ax = plt.subplots(
        figsize=(24, 21)
    )


    # --------------------------------------------------------
    # Heatmap
    # --------------------------------------------------------

    sns.heatmap(
        percentages,

        annot=annotations,

        fmt="",

        cmap="Blues",

        linewidths=0.5,

        linecolor="white",

        xticklabels=hierarchical_labels,

        yticklabels=hierarchical_labels,

        cbar_kws={
            "label":
            "Percentage (%)"
        },

        ax=ax,
    )


    # --------------------------------------------------------
    # Labels
    # --------------------------------------------------------

    ax.set_xlabel(
        "Predicted Label",
        fontsize=15,
        fontweight="bold",
    )


    ax.set_ylabel(
        "True Label",
        fontsize=15,
        fontweight="bold",
    )


    ax.set_title(
        title,
        fontsize=18,
        fontweight="bold",
        pad=20,
    )


    # --------------------------------------------------------
    # Tick labels
    # --------------------------------------------------------

    ax.set_xticklabels(
        hierarchical_labels,
        rotation=45,
        ha="right",
        fontsize=10,
    )


    ax.set_yticklabels(
        hierarchical_labels,
        rotation=0,
        fontsize=10,
    )


    # --------------------------------------------------------
    # Layout
    # --------------------------------------------------------

    plt.tight_layout()


    # --------------------------------------------------------
    # Save PNG
    # --------------------------------------------------------

    png_file = (
        CM_DIR
        / f"{prefix}.png"
    )


    plt.savefig(
        png_file,
        dpi=400,
        bbox_inches="tight",
    )


    # --------------------------------------------------------
    # Save PDF
    #
    # Useful for papers because it is vector-based.
    # --------------------------------------------------------

    pdf_file = (
        CM_DIR
        / f"{prefix}.pdf"
    )


    plt.savefig(
        pdf_file,
        bbox_inches="tight",
    )


    plt.close()


    print(
        "\nSaved:"
    )

    print(
        png_file
    )

    print(
        pdf_file
    )


In [45]:
def create_confusion_matrix(
    predictions,
    prefix,
    title,
):

    print("\n" + "-" * 100)

    print(
        "Creating:",
        title
    )


    # --------------------------------------------------------
    # Build count matrix
    # --------------------------------------------------------

    confusion = (
        build_label_confusion_matrix(
            test_labels,
            predictions,
        )
    )


    # --------------------------------------------------------
    # Convert to percentages
    # --------------------------------------------------------

    percentages = (
        confusion_to_percentages(
            confusion
        )
    )


    # --------------------------------------------------------
    # Save CSV
    # --------------------------------------------------------

    save_matrices(
        confusion,
        percentages,
        prefix,
    )


    # --------------------------------------------------------
    # Plot
    # --------------------------------------------------------

    plot_confusion_matrix(
        confusion,
        percentages,
        prefix,
        title,
    )


    return (
        confusion,
        percentages,
    )

In [46]:
# 1. ORIGINAL 0.5 THRESHOLD


print("\n" + "=" * 100)
print("1. ORIGINAL 0.5 THRESHOLD")
print("=" * 100)


predictions_05 = make_predictions(
    test_probabilities,
    0.5,
)


confusion_05, percentages_05 = (
    create_confusion_matrix(

        predictions=predictions_05,

        prefix="confusion_0.5",

        title=(
            "Hierarchical Multilabel "
            "Label Confusion Matrix "
            "(Threshold = 0.50)"
        ),
    )
)


1. ORIGINAL 0.5 THRESHOLD

----------------------------------------------------------------------------------------------------
Creating: Hierarchical Multilabel Label Confusion Matrix (Threshold = 0.50)

Saved:
C:\Users\alrazz\Downloads\Parent_Conditioned_Hierarchica\confusion_matrices\confusion_0.5.png
C:\Users\alrazz\Downloads\Parent_Conditioned_Hierarchica\confusion_matrices\confusion_0.5.pdf


In [47]:
# GLOBAL THRESHOLD TUNING

GLOBAL_TUNED_THRESHOLD = 0.28


print("\n" + "=" * 100)
print("2. GLOBAL TUNED THRESHOLD")
print("=" * 100)

print(
    "Global threshold:",
    GLOBAL_TUNED_THRESHOLD,
)


predictions_global = make_predictions(
    test_probabilities,
    GLOBAL_TUNED_THRESHOLD,
)


confusion_global, percentages_global = (
    create_confusion_matrix(

        predictions=predictions_global,

        prefix="confusion_global_tuned",

        title=(
            "Hierarchical Multilabel "
            "Label Confusion Matrix "
            f"(Global Threshold = "
            f"{GLOBAL_TUNED_THRESHOLD:.3f})"
        ),
    )
)


2. GLOBAL TUNED THRESHOLD
Global threshold: 0.28

----------------------------------------------------------------------------------------------------
Creating: Hierarchical Multilabel Label Confusion Matrix (Global Threshold = 0.280)

Saved:
C:\Users\alrazz\Downloads\Parent_Conditioned_Hierarchica\confusion_matrices\confusion_global_tuned.png
C:\Users\alrazz\Downloads\Parent_Conditioned_Hierarchica\confusion_matrices\confusion_global_tuned.pdf


In [48]:
#  PER-LABEL THRESHOLD TUNING

PER_LABEL_THRESHOLDS = {

"MT"  : 0.11,
"LY"  : 0.23,
"SP"  : 0.36,
"ID"  : 0.26,
"NA"  : 0.40,
"HI"  : 0.39,
"IN"  : 0.52,
"OP"  : 0.39,
"IP"  : 0.41,
"it"  : 0.31,
"ne"  : 0.40,
"sr"  : 0.33,
"nb"  : 0.13,
"re"  : 0.28,
"en"  : 0.50,
"ra"  : 0.10,
"dtp" : 0.33,
"fi"  : 0.53,
"lt"  : 0.28,
"rv"  : 0.23,
"ob"  : 0.30,
"rs"  : 0.39,
"av"  : 0.60,
"ds"  : 0.80,
"ed"  : 0.24

}


In [49]:
per_label_threshold_array = np.array(
    [
        PER_LABEL_THRESHOLDS[label]
        for label in hierarchical_labels
    ],
    dtype=np.float32,
)


print("\nPer-label thresholds:")

for label, threshold in zip(
    hierarchical_labels,
    per_label_threshold_array,
):

    print(
        f"{label:>5s}: "
        f"{threshold:.4f}"
    )


Per-label thresholds:
   MT: 0.1100
   LY: 0.2300
   SP: 0.3600
   ID: 0.2600
   NA: 0.4000
   HI: 0.3900
   IN: 0.5200
   OP: 0.3900
   IP: 0.4100
   it: 0.3100
   ne: 0.4000
   sr: 0.3300
   nb: 0.1300
   re: 0.2800
   en: 0.5000
   ra: 0.1000
  dtp: 0.3300
   fi: 0.5300
   lt: 0.2800
   rv: 0.2300
   ob: 0.3000
   rs: 0.3900
   av: 0.6000
   ds: 0.8000
   ed: 0.2400


In [50]:
predictions_per_label = (
    make_predictions(
        test_probabilities,
        per_label_threshold_array,
    )
)


confusion_per_label, percentages_per_label = (
    create_confusion_matrix(

        predictions=predictions_per_label,

        prefix="confusion_per_label_tuned",

        title=(
            "Hierarchical Multilabel "
            "Label Confusion Matrix "
            "(Per-Label Tuned Thresholds)"
        ),
    )
)




----------------------------------------------------------------------------------------------------
Creating: Hierarchical Multilabel Label Confusion Matrix (Per-Label Tuned Thresholds)

Saved:
C:\Users\alrazz\Downloads\Parent_Conditioned_Hierarchica\confusion_matrices\confusion_per_label_tuned.png
C:\Users\alrazz\Downloads\Parent_Conditioned_Hierarchica\confusion_matrices\confusion_per_label_tuned.pdf


## Confusion Matrix Parent Child seprately

In [51]:

def create_subset_confusion_matrix(
    true_labels,
    predicted_labels,
    label_indices,
    label_names,
    prefix,
    title,
):


    # --------------------------------------------------------
    # Extract only the desired labels
    # --------------------------------------------------------

    true_subset = (
        true_labels[:, label_indices]
    )

    predicted_subset = (
        predicted_labels[:, label_indices]
    )


    n_labels = len(
        label_names
    )


    # --------------------------------------------------------
    # Create confusion matrix
    # --------------------------------------------------------

    confusion = np.zeros(
        (n_labels, n_labels),
        dtype=np.int64,
    )


    # --------------------------------------------------------
    # Count true -> predicted label associations
    # --------------------------------------------------------

    for i in range(
        true_subset.shape[0]
    ):

        true_indices = np.where(
            true_subset[i] == 1
        )[0]

        predicted_indices = np.where(
            predicted_subset[i] == 1
        )[0]


        for true_idx in true_indices:

            for pred_idx in predicted_indices:

                confusion[
                    true_idx,
                    pred_idx
                ] += 1


    # --------------------------------------------------------
    # Row-wise percentages
    # --------------------------------------------------------

    row_totals = (
        confusion.sum(axis=1)
    )


    percentages = np.zeros_like(
        confusion,
        dtype=np.float64,
    )


    for i in range(n_labels):

        if row_totals[i] > 0:

            percentages[i] = (
                confusion[i]
                / row_totals[i]
                * 100.0
            )


    # ========================================================
    # SAVE COUNTS
    # ========================================================

    count_df = pd.DataFrame(
        confusion,
        index=label_names,
        columns=label_names,
    )


    count_file = (
        CM_DIR
        / f"{prefix}_counts.csv"
    )


    count_df.to_csv(
        count_file,
        encoding="utf-8-sig",
    )


    # ========================================================
    # SAVE PERCENTAGES
    # ========================================================

    percent_df = pd.DataFrame(
        percentages,
        index=label_names,
        columns=label_names,
    )


    percent_file = (
        CM_DIR
        / f"{prefix}_percentages.csv"
    )


    percent_df.to_csv(
        percent_file,
        encoding="utf-8-sig",
        float_format="%.4f",
    )


    # ========================================================
    # CREATE ANNOTATIONS
    # ========================================================

    annotations = np.empty(
        confusion.shape,
        dtype=object,
    )


    for i in range(n_labels):

        for j in range(n_labels):

            annotations[i, j] = (
                f"{confusion[i, j]:,}\n"
                f"{percentages[i, j]:.1f}%"
            )


    # ========================================================
    # FIGURE SIZE
    # ========================================================

    if n_labels <= 10:

        figsize = (12, 10)

    elif n_labels <= 16:

        figsize = (16, 14)

    else:

        figsize = (24, 21)


    fig, ax = plt.subplots(
        figsize=figsize
    )


    # ========================================================
    # HEATMAP
    # ========================================================

    sns.heatmap(

        percentages,

        annot=annotations,

        fmt="",

        cmap="Blues",

        linewidths=0.6,

        linecolor="white",

        xticklabels=label_names,

        yticklabels=label_names,

        cbar_kws={
            "label":
            "Percentage (%)"
        },

        ax=ax,
    )


    # ========================================================
    # AXIS LABELS
    # ========================================================

    ax.set_xlabel(
        "Predicted Label",
        fontsize=14,
        fontweight="bold",
    )


    ax.set_ylabel(
        "True Label",
        fontsize=14,
        fontweight="bold",
    )


    ax.set_title(
        title,
        fontsize=17,
        fontweight="bold",
        pad=18,
    )


    # ========================================================
    # TICK LABELS
    # ========================================================

    ax.set_xticklabels(
        label_names,
        rotation=45,
        ha="right",
        fontsize=11,
    )


    ax.set_yticklabels(
        label_names,
        rotation=0,
        fontsize=11,
    )


    # ========================================================
    # SAVE PNG
    # ========================================================

    png_file = (
        CM_DIR
        / f"{prefix}.png"
    )


    plt.tight_layout()


    plt.savefig(
        png_file,
        dpi=400,
        bbox_inches="tight",
    )


    # ========================================================
    # SAVE PDF
    # ========================================================

    pdf_file = (
        CM_DIR
        / f"{prefix}.pdf"
    )


    plt.savefig(
        pdf_file,
        bbox_inches="tight",
    )


    plt.close()


    print(
        "\nSaved:"
    )

    print(
        png_file
    )

    print(
        pdf_file
    )

    print(
        count_file
    )

    print(
        percent_file
    )


    return (
        confusion,
        percentages,
    )





In [52]:
# ============================================================
# LABEL INDICES
# ============================================================

parent_indices = list(
    range(
        len(parents)
    )
)


child_indices = list(
    range(
        len(parents),
        len(hierarchical_labels)
    )
)


In [53]:

print("\nParent indices:")
print(parent_indices)


print("\nChild indices:")
print(child_indices)


Parent indices:
[0, 1, 2, 3, 4, 5, 6, 7, 8]

Child indices:
[9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24]


In [54]:
def create_parent_child_matrices(
    predictions,
    prefix,
    threshold_description,
):

    # ========================================================
    # PARENT MATRIX
    # ========================================================

    print("\n" + "-" * 100)

    print(
        "Creating parent confusion matrix:",
        threshold_description,
    )


    parent_confusion, parent_percentages = (
        create_subset_confusion_matrix(

            true_labels=test_labels,

            predicted_labels=predictions,

            label_indices=parent_indices,

            label_names=parents,

            prefix=(
                f"{prefix}_parents"
            ),

            title=(
                "Parent Label Confusion Matrix\n"
                f"{threshold_description}"
            ),
        )
    )


    # ========================================================
    # CHILD MATRIX
    # ========================================================

    print("\n" + "-" * 100)

    print(
        "Creating child confusion matrix:",
        threshold_description,
    )


    child_confusion, child_percentages = (
        create_subset_confusion_matrix(

            true_labels=test_labels,

            predicted_labels=predictions,

            label_indices=child_indices,

            label_names=children,

            prefix=(
                f"{prefix}_children"
            ),

            title=(
                "Child Label Confusion Matrix\n"
                f"{threshold_description}"
            ),
        )
    )


    return {
        "parent_confusion":
            parent_confusion,

        "parent_percentages":
            parent_percentages,

        "child_confusion":
            child_confusion,

        "child_percentages":
            child_percentages,
    }

In [55]:
# ORIGINAL 0.5 THRESHOLD
print("\n" + "=" * 100)
print("PARENT + CHILD MATRICES — THRESHOLD 0.50")
print("=" * 100)


parent_child_05 = (
    create_parent_child_matrices(

        predictions=predictions_05,

        prefix="confusion_0.5",

        threshold_description=(
            "Threshold = 0.50"
        ),
    )
)



PARENT + CHILD MATRICES — THRESHOLD 0.50

----------------------------------------------------------------------------------------------------
Creating parent confusion matrix: Threshold = 0.50

Saved:
C:\Users\alrazz\Downloads\Parent_Conditioned_Hierarchica\confusion_matrices\confusion_0.5_parents.png
C:\Users\alrazz\Downloads\Parent_Conditioned_Hierarchica\confusion_matrices\confusion_0.5_parents.pdf
C:\Users\alrazz\Downloads\Parent_Conditioned_Hierarchica\confusion_matrices\confusion_0.5_parents_counts.csv
C:\Users\alrazz\Downloads\Parent_Conditioned_Hierarchica\confusion_matrices\confusion_0.5_parents_percentages.csv

----------------------------------------------------------------------------------------------------
Creating child confusion matrix: Threshold = 0.50

Saved:
C:\Users\alrazz\Downloads\Parent_Conditioned_Hierarchica\confusion_matrices\confusion_0.5_children.png
C:\Users\alrazz\Downloads\Parent_Conditioned_Hierarchica\confusion_matrices\confusion_0.5_children.pdf
C:\U

In [56]:
#Global threshold
print("\n" + "=" * 100)
print("PARENT + CHILD MATRICES — GLOBAL TUNED")
print("=" * 100)


parent_child_global = (
    create_parent_child_matrices(

        predictions=predictions_global,

        prefix="confusion_global_tuned",

        threshold_description=(
            "Global Tuned Threshold = "
            f"{GLOBAL_TUNED_THRESHOLD:.3f}"
        ),
    )
)



PARENT + CHILD MATRICES — GLOBAL TUNED

----------------------------------------------------------------------------------------------------
Creating parent confusion matrix: Global Tuned Threshold = 0.280

Saved:
C:\Users\alrazz\Downloads\Parent_Conditioned_Hierarchica\confusion_matrices\confusion_global_tuned_parents.png
C:\Users\alrazz\Downloads\Parent_Conditioned_Hierarchica\confusion_matrices\confusion_global_tuned_parents.pdf
C:\Users\alrazz\Downloads\Parent_Conditioned_Hierarchica\confusion_matrices\confusion_global_tuned_parents_counts.csv
C:\Users\alrazz\Downloads\Parent_Conditioned_Hierarchica\confusion_matrices\confusion_global_tuned_parents_percentages.csv

----------------------------------------------------------------------------------------------------
Creating child confusion matrix: Global Tuned Threshold = 0.280

Saved:
C:\Users\alrazz\Downloads\Parent_Conditioned_Hierarchica\confusion_matrices\confusion_global_tuned_children.png
C:\Users\alrazz\Downloads\Parent_Con

In [57]:
# PER-LABEL TUNED THRESHOLDS


print("\n" + "=" * 100)
print("PARENT + CHILD MATRICES — PER-LABEL TUNED")
print("=" * 100)


parent_child_per_label = (
    create_parent_child_matrices(

        predictions=predictions_per_label,

        prefix="confusion_per_label_tuned",

        threshold_description=(
            "Per-Label Tuned Thresholds"
        ),
    )
)



PARENT + CHILD MATRICES — PER-LABEL TUNED

----------------------------------------------------------------------------------------------------
Creating parent confusion matrix: Per-Label Tuned Thresholds

Saved:
C:\Users\alrazz\Downloads\Parent_Conditioned_Hierarchica\confusion_matrices\confusion_per_label_tuned_parents.png
C:\Users\alrazz\Downloads\Parent_Conditioned_Hierarchica\confusion_matrices\confusion_per_label_tuned_parents.pdf
C:\Users\alrazz\Downloads\Parent_Conditioned_Hierarchica\confusion_matrices\confusion_per_label_tuned_parents_counts.csv
C:\Users\alrazz\Downloads\Parent_Conditioned_Hierarchica\confusion_matrices\confusion_per_label_tuned_parents_percentages.csv

----------------------------------------------------------------------------------------------------
Creating child confusion matrix: Per-Label Tuned Thresholds

Saved:
C:\Users\alrazz\Downloads\Parent_Conditioned_Hierarchica\confusion_matrices\confusion_per_label_tuned_children.png
C:\Users\alrazz\Downloads\